# Caso C — AOV · Modelado y evaluación

> **Objetivo.** Explicar qué mueve el ticket (modelo inferencial) y predecir el gasto del cliente recurrente (modelo predictivo).

> **Entradas.** Tabla maestra de tickets enriquecidos (loyalty, exógenas, promos), vía catálogo.

> **Salidas.** Coeficientes del GLM con significancia, métricas del predictivo, comparación de modelos, figuras de desempeño e importancia SHAP.

> **Cómo ejecutar.** `Restart & Run All`; determinista. Corre en paralelo al pipeline `caso_c`.

## 1. Datos: tickets enriquecidos

Cruce transacciones ⨝ loyalty ⨝ exógenas ⨝ promociones.

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_c, _ = masters.build_master_c(
        catalog.load('c_transacciones_resumen'), catalog.load('c_clientes_loyalty'),
        catalog.load('c_variables_exogenas'), catalog.load('c_promociones_activas'))
print('Master:', master_c.shape)
master_c.head()

## 2. Estrategia: dos modelos complementarios

- **Inferencial — GLM gaussiano:** coeficientes con error estándar, p-valor e IC95%; outliers **winsorizados**. Responde *qué* mueve el ticket, con significancia.
- **Predictivo — boosting sobre RFM+loyalty:** holdout **80/20** sobre clientes recurrentes, HPO con Optuna (K-Fold); se comparan Ridge/GBR/ensemble.

In [ ]:
from tostao_ml.cases.caso_c import run_case_c

result = run_case_c(master_c, tune=True)

## 3. Drivers del ticket (GLM inferencial)

Un coeficiente es un efecto real si es significativo (p<0.05) y su IC95% no cruza cero.

In [ ]:
result.coefficients.round(4)

In [ ]:
import plotly.graph_objects as go

coefs = result.coefficients
top = coefs.reindex(coefs['coef'].abs().sort_values().index).tail(12)
fig = go.Figure(go.Bar(x=top['coef'], y=list(top.index), orientation='h'))
fig.add_vline(x=0, line_dash='dash')
fig.update_layout(title='Drivers del ticket (β)', height=420)
fig.show()

## 4. Modelo predictivo del gasto (holdout)

R² fuerte y mejora sobre el baseline de la media confirman poder predictivo real.

In [ ]:
import pandas as pd

pd.DataFrame([result.predictive_metrics]).T.rename(columns={0: 'valor'}).round(4)

In [ ]:
if result.comparison is not None:
    display(result.comparison.round(4))

In [ ]:
from tostao_ml.framework.evaluation import performance

test = result.predictive_test
performance.pred_vs_actual(test['ticket_medio'], test['pred']).show()
performance.residuals_vs_pred(test['ticket_medio'], test['pred']).show()

## 5. Interpretabilidad (SHAP)

Contribución media de cada feature al gasto predicho.

In [ ]:
from tostao_ml.framework.interpret import compute_shap, shap_summary_bar

if result.predictive_model is not None and result.predictive_features is not None:
    shap_res = compute_shap(result.predictive_model, result.predictive_features, sample_size=120)
    shap_summary_bar(shap_res).show()

## 6. Lectura interpretada

Significancia de los drivers y calidad de la predicción, en palabras.

In [ ]:
from IPython.display import Markdown
from tostao_ml.cases import storytelling as st

Markdown(st.interpret_model_c(result).to_markdown())

## Conclusión

`total_articulos` es el driver dominante y altamente significativo; el clima lluvioso reduce el ticket con efecto pequeño. El modelo de gasto (R² ~0.81) predice bien y prioriza clientes por valor esperado para campañas.